# Review and validate v2 data
This notebook reads only the frozen v2 files. It checks counts, global prompt separation, IDs, token lengths, provenance, and representative examples.

In [1]:
import os, sys, json, subprocess
if os.path.exists('/content'):
    if not os.path.exists('/content/mfr-dpo'):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    else:
        data_status = subprocess.run(
            ['git', '-C', '/content/mfr-dpo', 'status', '--porcelain', '--', 'data/v2'],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if data_status:
            print('Using the existing data/v2; skipping git pull so local data is not overwritten.')
        else:
            !git -C /content/mfr-dpo pull --ff-only -q
    REPO = '/content/mfr-dpo'
else:
    REPO = '..'
sys.path.insert(0, f'{REPO}/src')
import mfr_data
splits = mfr_data.load_splits(f'{REPO}/data/v2')
print(json.dumps(json.load(open(f'{REPO}/data/v2/manifest.json')), indent=2)[:4000])


{
  "created_at_utc": "2026-09-13T18:38:55.594609+00:00",
  "data_version": "v2",
  "files": {
    "helpful_test.jsonl": {
      "rows": 300,
      "sha256": "922042041624a37eea7f58abae788a17569a9eadbace6f3b44a531bd99cc7b77"
    },
    "helpful_train.jsonl": {
      "rows": 2000,
      "sha256": "c89c470f4501e640de71aac723210907e3fd376df93ac3b51226b2a263f302ea"
    },
    "helpful_val.jsonl": {
      "rows": 200,
      "sha256": "58d4b2b58e873b591c13ab7438d54ad4ca36752c5399c57722891702e485554c"
    },
    "quality_test.jsonl": {
      "rows": 300,
      "sha256": "3c82641b1e0d2286a84fac90417c680d9ad9536c780b814102fed4404724b662"
    },
    "quality_train.jsonl": {
      "rows": 2000,
      "sha256": "dbbb832dc55587a80bd60cb6e38528c52c7526bc930ab76da12eafaa003daa0f"
    },
    "quality_val.jsonl": {
      "rows": 200,
      "sha256": "55ea0b197dbe1653dcf6751d4ae7857fd6ddd3d3461dce0f57972112c5d0deab"
    },
    "safe_test.jsonl": {
      "rows": 300,
      "sha256": "11a4358ef261f817f210

In [2]:
mfr_data.validate_splits(splits, max_tokens=1024, expected_sizes={'train': 2000, 'val': 200, 'test': 300})
for dataset, parts in splits.items():
    for split, frame in parts.items():
        longest = frame.prompt_tokens + frame[['chosen_tokens','rejected_tokens']].max(axis=1)
        print(f'{dataset:8s} {split:5s}: {len(frame):4d} rows, max {longest.max():4d} tokens')


helpful  train: 2000 rows, max 1024 tokens
helpful  val  :  200 rows, max 1018 tokens
helpful  test :  300 rows, max 1014 tokens
safe     train: 2000 rows, max  881 tokens
safe     val  :  200 rows, max  594 tokens
safe     test :  300 rows, max  452 tokens
quality  train: 2000 rows, max 1023 tokens
quality  val  :  200 rows, max 1020 tokens
quality  test :  300 rows, max 1022 tokens


In [3]:
for dataset in splits:
    row = splits[dataset]['train'].sample(1, random_state=0).iloc[0]
    print('\n', dataset.upper(), row['id'], row['source_dataset'], row['source_row_id'])
    print('PROMPT:', row.prompt[:500])
    print('CHOSEN:', row.chosen[:500])
    print('REJECTED:', row.rejected[:500])



 HELPFUL helpful-train-0405 nvidia/HelpSteer2 6524
PROMPT: Summarize the article in the reference text using a table.

Reference text:
TYPES OF SUIT FABRIC: 4 COMMON TYPES
Step into the world of fabric suits and explore the incredible range of options that will take your style game to new heights! From the sophisticated elegance of satin to the luxurious comfort of velvet, each fabric brings its own unique characteristics. Let’s dive into the most common suit fabrics and uncover the secrets behind these magnificent materials.
LINEN FABRIC
Linen fabric i
CHOSEN: | Fabric    | Characteristics                                                                                                                                                                                                                                                                                                                                           | Cost                                                                 